# DX 704 Week 6 Project

This project will develop a treatment plan for a fictious illness "Twizzleflu".
Twizzleflu is a mild illness caused by a virus.
The main symptoms are a mild fever, fidgeting, and kicking the blankets off the bed or couch.
Mild dehydration has also been reported in more severe cases.
These symptoms typically last 1-2 weeks without treatment.
Word on the internet says that Twizzleflu can be cured faster by drinking copious orange juice, but this has not been supported by evidence so far.
You will be provided with a theoretical model of Twizzleflu modeled as a Markov decision process.
Based on the model, you will compute optimal treatment plans to optimize different criteria, and compare patient discomfort with the different plans.

The full project description, a template notebook, and raw data are available on GitHub: [Project 6 Materials](https://github.com/bu-cds-dx704/dx704-project-06).

We will model Twizzleflu as a Markov decision process.
The model transition probabilities are provided in the file "twizzleflu-transitions.tsv" and the expected rewards are in "twizzleflu-rewards.tsv".
The goal for Twizzleflu is to minimize the expected discomfort of the patient which is expressed as negative rewards in the file.

## Example Code

You may find it helpful to refer to these GitHub repositories of Jupyter notebooks for example code.

* https://github.com/bu-cds-omds/dx601-examples
* https://github.com/bu-cds-omds/dx602-examples
* https://github.com/bu-cds-omds/dx603-examples
* https://github.com/bu-cds-omds/dx704-examples

Any calculations demonstrated in code examples or videos may be found in these notebooks, and you are allowed to copy this example code in your homework answers.

## Part 1: Evaluate a Do Nothing Plan

One of the treatment actions is to do nothing.
Calculate the expected discomfort (not rewards) of a policy that always does nothing.

Hint: for this value calculation and later ones, use value iteration.
The analytical solution has difficulties in practice when there is no discount factor.

In [2]:
%pip install numpy pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 42.2 MB/s  0:00:006m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 42.7 MB/s  0:00:006m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [pandas]2m1/2 [pandas]
Note: you may need to restart the kernel to use updated packages.


In [3]:
# YOUR CHANGES HERE
import numpy as np
import pandas as pd

# Load the model's transition probabilities and rewards
transitions = pd.read_csv("twizzleflu-transitions.tsv", sep="\t")
rewards = pd.read_csv("twizzleflu-rewards.tsv", sep="\t")

# Keep only the rows for the do-nothing action
nothing_transitions = transitions[
    transitions["action"] == "do-nothing"
].copy()

nothing_rewards = rewards[
    rewards["action"] == "do-nothing"
].copy()

display(nothing_transitions)
display(nothing_rewards)


,action,state,next_state,probability
0,do-nothing,exposed-1,exposed-2,0.8
1,do-nothing,exposed-1,recovered,0.2
2,do-nothing,exposed-2,exposed-3,0.8
3,do-nothing,exposed-2,recovered,0.2
4,do-nothing,exposed-3,symptoms-1,0.8
5,do-nothing,exposed-3,recovered,0.2
6,do-nothing,symptoms-1,symptoms-1,0.7
7,do-nothing,symptoms-1,symptoms-2,0.3
8,do-nothing,symptoms-2,symptoms-2,0.7
9,do-nothing,symptoms-2,symptoms-3,0.3


,action,state,reward
0,do-nothing,exposed-1,0.0
1,do-nothing,exposed-2,0.0
2,do-nothing,exposed-3,0.0
3,do-nothing,symptoms-1,-0.5
4,do-nothing,symptoms-2,-1.0
5,do-nothing,symptoms-3,-0.5
6,do-nothing,recovered,0.0


In [4]:
# List all states
states = nothing_rewards["state"].tolist()

# Convert negative rewards into positive daily discomfort
daily_discomfort = (
    nothing_rewards.set_index("state")["reward"] * -1
).to_dict()

# Start with zero estimated total discomfort for every state
discomfort = {state: 0.0 for state in states}

tolerance = 1e-10

for iteration in range(10000):
    new_discomfort = {}

    for state in states:
        # Possible next states when doing nothing
        state_transitions = nothing_transitions[
            nothing_transitions["state"] == state
        ]

        # Probability-weighted future discomfort
        future_discomfort = sum(
            row.probability * discomfort[row.next_state]
            for row in state_transitions.itertuples(index=False)
        )

        new_discomfort[state] = (
            daily_discomfort[state] + future_discomfort
        )

    # Measure the largest change across states
    largest_change = max(
        abs(new_discomfort[state] - discomfort[state])
        for state in states
    )

    discomfort = new_discomfort

    if largest_change < tolerance:
        print(f"Converged after {iteration + 1} iterations.")
        break
else:
    raise RuntimeError("Value iteration did not converge.")

# Organize the results
do_nothing_results = pd.DataFrame({
    "state": states,
    "expected_discomfort": [discomfort[state] for state in states]
})

display(do_nothing_results)

Converged after 83 iterations.


,state,expected_discomfort
0,exposed-1,3.413333
1,exposed-2,4.266667
2,exposed-3,5.333333
3,symptoms-1,6.666667
4,symptoms-2,5.000000
5,symptoms-3,1.666667
6,recovered,0.000000


Save the expected discomfort by state to a file "do-nothing-discomfort.tsv" with columns state and expected_discomfort.

In [5]:
# YOUR CHANGES HERE

do_nothing_results.to_csv(
    "do-nothing-discomfort.tsv",
    sep="\t",
    index=False
)

Submit "do-nothing-discomfort.tsv" in Gradescope.

## Part 2: Compute an Optimal Treatment Plan

Compute an optimal treatment plan for Twizzleflu.
It should minimize the expected discomfort (maximize the rewards).

In [6]:
# YOUR CHANGES HERE

# List all available actions
actions = rewards["action"].drop_duplicates().tolist()

# Look up daily discomfort using (state, action)
discomfort_lookup = (
    -rewards.set_index(["state", "action"])["reward"]
).to_dict()

# Start with zero estimated discomfort for every state
optimal_discomfort = {state: 0.0 for state in states}

tolerance = 1e-10

for iteration in range(10000):
    new_discomfort = {}

    for state in states:
        action_costs = {}

        for action in actions:
            # Possible next states for this state and action
            possible_transitions = transitions[
                (transitions["state"] == state)
                & (transitions["action"] == action)
            ]

            future_discomfort = sum(
                row.probability * optimal_discomfort[row.next_state]
                for row in possible_transitions.itertuples(index=False)
            )

            action_costs[action] = (
                discomfort_lookup[(state, action)]
                + future_discomfort
            )

        # Keep the smallest expected discomfort
        new_discomfort[state] = min(action_costs.values())

    largest_change = max(
        abs(new_discomfort[state] - optimal_discomfort[state])
        for state in states
    )

    optimal_discomfort = new_discomfort

    if largest_change < tolerance:
        print(f"Converged after {iteration + 1} iterations.")
        break
else:
    raise RuntimeError("Value iteration did not converge.")

# Choose actions using the converged discomfort estimates
optimal_policy = {}

for state in states:
    action_costs = {}

    for action in actions:
        possible_transitions = transitions[
            (transitions["state"] == state)
            & (transitions["action"] == action)
        ]

        future_discomfort = sum(
            row.probability * optimal_discomfort[row.next_state]
            for row in possible_transitions.itertuples(index=False)
        )

        action_costs[action] = (
            discomfort_lookup[(state, action)]
            + future_discomfort
        )

    optimal_policy[state] = min(action_costs, key=action_costs.get)

# Display the optimal treatment plan
minimum_discomfort_actions = pd.DataFrame({
    "state": states,
    "action": [optimal_policy[state] for state in states]
})

display(minimum_discomfort_actions)

Converged after 100 iterations.


,state,action
0,exposed-1,sleep-8
1,exposed-2,sleep-8
2,exposed-3,sleep-8
3,symptoms-1,drink-oj
4,symptoms-2,drink-oj
5,symptoms-3,drink-oj
6,recovered,do-nothing


Save the optimal actions for each state to a file "minimum-discomfort-actions.tsv" with columns state and action.

In [7]:
# YOUR CHANGES HERE

minimum_discomfort_actions.to_csv(
    "minimum-discomfort-actions.tsv",
    sep="\t",
    index=False
)

Submit "minimum-discomfort-actions.tsv" in Gradescope.

## Part 3: Expected Discomfort

Using your previous optimal policy, compute the expected discomfort for each state.

In [8]:
# YOUR CHANGES HERE

# Start with zero estimated discomfort
policy_discomfort = {state: 0.0 for state in states}

for iteration in range(10000):
    new_discomfort = {}

    for state in states:
        # Follow the optimal treatment plan from Part 2
        action = optimal_policy[state]

        possible_transitions = transitions[
            (transitions["state"] == state)
            & (transitions["action"] == action)
        ]

        future_discomfort = sum(
            row.probability * policy_discomfort[row.next_state]
            for row in possible_transitions.itertuples(index=False)
        )

        new_discomfort[state] = (
            discomfort_lookup[(state, action)]
            + future_discomfort
        )

    largest_change = max(
        abs(new_discomfort[state] - policy_discomfort[state])
        for state in states
    )

    policy_discomfort = new_discomfort

    if largest_change < tolerance:
        print(f"Converged after {iteration + 1} iterations.")
        break
else:
    raise RuntimeError("Policy evaluation did not converge.")

minimum_discomfort_values = pd.DataFrame({
    "state": states,
    "expected_discomfort": [
        policy_discomfort[state] for state in states
    ]
})

display(minimum_discomfort_values)

# These should agree with the optimal values from Part 2
assert np.allclose(
    [policy_discomfort[state] for state in states],
    [optimal_discomfort[state] for state in states]
)

Converged after 100 iterations.


,state,expected_discomfort
0,exposed-1,0.75
1,exposed-2,1.50
2,exposed-3,3.00
3,symptoms-1,6.00
4,symptoms-2,4.50
5,symptoms-3,1.50
6,recovered,0.00


Save your results in a file "minimum-discomfort-values.tsv" with columns state and expected_discomfort.

In [9]:
# YOUR CHANGES HERE

minimum_discomfort_values.to_csv(
    "minimum-discomfort-values.tsv",
    sep="\t",
    index=False
)

Submit "minimum-discomfort-values.tsv" in Gradescope.

## Part 4: Minimizing Twizzleflu Duration

Modifiy the Markov decision process to minimize the days until the Twizzle flu is over.
To do so, change the reward function to always be -1 if the current state corresponds to being sick (must have symptoms, exposed does not count) and 0 otherwise.
To be clear, the action does not matter for this reward function.


In [10]:
# YOUR CHANGES HERE

# Copy the original table so we can still use its discomfort rewards later
duration_rewards = rewards.copy()

# Assign -1 to symptomatic states and 0 to all other states
duration_rewards["reward"] = np.where(
    duration_rewards["state"].str.startswith("symptoms-"),
    -1.0,
    0.0
)

display(duration_rewards)

,action,state,reward
0,do-nothing,exposed-1,0.0
1,do-nothing,exposed-2,0.0
2,do-nothing,exposed-3,0.0
3,do-nothing,symptoms-1,-1.0
4,do-nothing,symptoms-2,-1.0
5,do-nothing,symptoms-3,-1.0
6,do-nothing,recovered,0.0
7,drink-oj,exposed-1,0.0
8,drink-oj,exposed-2,0.0
9,drink-oj,exposed-3,0.0


In [11]:
# Copy the original table so we can still use its discomfort rewards later
duration_rewards = rewards.copy()

# Assign -1 to symptomatic states and 0 to all other states
duration_rewards["reward"] = np.where(
    duration_rewards["state"].str.startswith("symptoms-"),
    -1.0,
    0.0
)

display(duration_rewards)

,action,state,reward
0,do-nothing,exposed-1,0.0
1,do-nothing,exposed-2,0.0
2,do-nothing,exposed-3,0.0
3,do-nothing,symptoms-1,-1.0
4,do-nothing,symptoms-2,-1.0
5,do-nothing,symptoms-3,-1.0
6,do-nothing,recovered,0.0
7,drink-oj,exposed-1,0.0
8,drink-oj,exposed-2,0.0
9,drink-oj,exposed-3,0.0


Save your new reward function in a file "duration-rewards.tsv" in the same format as "twizzleflu-rewards.tsv".

In [12]:
# YOUR CHANGES HERE

duration_rewards.to_csv(
    "duration-rewards.tsv",
    sep="\t",
    index=False
)

Submit "duration-rewards.tsv" in Gradescope.

## Part 5: Optimize for Shorter Twizzleflu

Compute an optimal policy to minimize the duration of Twizzleflu.

In [13]:
# YOUR CHANGES HERE

# Convert duration rewards into daily sick-day costs
sick_day_lookup = (
    -duration_rewards.set_index(["state", "action"])["reward"]
).to_dict()

# Start with zero estimated sick days
optimal_sick_days = {state: 0.0 for state in states}

for iteration in range(10000):
    new_sick_days = {}

    for state in states:
        action_costs = {}

        for action in actions:
            possible_transitions = transitions[
                (transitions["state"] == state)
                & (transitions["action"] == action)
            ]

            future_sick_days = sum(
                row.probability * optimal_sick_days[row.next_state]
                for row in possible_transitions.itertuples(index=False)
            )

            action_costs[action] = (
                sick_day_lookup[(state, action)]
                + future_sick_days
            )

        new_sick_days[state] = min(action_costs.values())

    largest_change = max(
        abs(new_sick_days[state] - optimal_sick_days[state])
        for state in states
    )

    optimal_sick_days = new_sick_days

    if largest_change < tolerance:
        print(f"Converged after {iteration + 1} iterations.")
        break
else:
    raise RuntimeError("Value iteration did not converge.")

# Choose actions using the converged estimates
duration_policy = {}

for state in states:
    action_costs = {}

    for action in actions:
        possible_transitions = transitions[
            (transitions["state"] == state)
            & (transitions["action"] == action)
        ]

        future_sick_days = sum(
            row.probability * optimal_sick_days[row.next_state]
            for row in possible_transitions.itertuples(index=False)
        )

        action_costs[action] = (
            sick_day_lookup[(state, action)]
            + future_sick_days
        )

    duration_policy[state] = min(action_costs, key=action_costs.get)

minimum_duration_actions = pd.DataFrame({
    "state": states,
    "action": [duration_policy[state] for state in states]
})

display(minimum_duration_actions)

Converged after 84 iterations.


,state,action
0,exposed-1,sleep-8
1,exposed-2,sleep-8
2,exposed-3,sleep-8
3,symptoms-1,do-nothing
4,symptoms-2,do-nothing
5,symptoms-3,do-nothing
6,recovered,do-nothing


Save the optimal actions for each state to a file "minimum-duration-actions.tsv" with columns state and action.

In [14]:
# YOUR CHANGES HERE

minimum_duration_actions.to_csv(
    "minimum-duration-actions.tsv",
    sep="\t",
    index=False
)

Submit "minimum-duration-actions.tsv" in Gradescope.

## Part 6: Shorter Twizzleflu?

Compute the expected number of days sick for each state to a file.

In [15]:
# YOUR CHANGES HERE

# Start with zero estimated sick days
policy_sick_days = {state: 0.0 for state in states}

for iteration in range(10000):
    new_sick_days = {}

    for state in states:
        # Follow the fastest policy from Part 5
        action = duration_policy[state]

        possible_transitions = transitions[
            (transitions["state"] == state)
            & (transitions["action"] == action)
        ]

        future_sick_days = sum(
            row.probability * policy_sick_days[row.next_state]
            for row in possible_transitions.itertuples(index=False)
        )

        new_sick_days[state] = (
            sick_day_lookup[(state, action)]
            + future_sick_days
        )

    largest_change = max(
        abs(new_sick_days[state] - policy_sick_days[state])
        for state in states
    )

    policy_sick_days = new_sick_days

    if largest_change < tolerance:
        print(f"Converged after {iteration + 1} iterations.")
        break
else:
    raise RuntimeError("Policy evaluation did not converge.")

minimum_duration_days = pd.DataFrame({
    "state": states,
    "expected_sick_days": [
        policy_sick_days[state] for state in states
    ]
})

display(minimum_duration_days)

# Check agreement with Part 5's optimal values
assert np.allclose(
    [policy_sick_days[state] for state in states],
    [optimal_sick_days[state] for state in states]
)

Converged after 84 iterations.


,state,expected_sick_days
0,exposed-1,1.250000
1,exposed-2,2.500000
2,exposed-3,5.000000
3,symptoms-1,10.000000
4,symptoms-2,6.666667
5,symptoms-3,3.333333
6,recovered,0.000000


Save the expected sick days for each state to a file "minimum-duration-days.tsv" with columns state and expected_sick_days.

In [16]:
# YOUR CHANGES HERE

minimum_duration_days.to_csv(
    "minimum-duration-days.tsv",
    sep="\t",
    index=False
)

Submit "minimum-duration-days.tsv" in Gradescope.

## Part 7: Speed vs Pampering

Compute the expected discomfort using the policy to minimize days sick, and compare the results to the expected discomfort when optimizing to minimize discomfort.

Hint: Make sure that you are reporting on expected discomfort, not repeating the expected number of days.

In [17]:
# YOUR CHANGES HERE

# Estimate discomfort when following the fastest policy
speed_discomfort = {state: 0.0 for state in states}

for iteration in range(10000):
    new_discomfort = {}

    for state in states:
        # Choose the action that minimizes sick days
        action = duration_policy[state]

        possible_transitions = transitions[
            (transitions["state"] == state)
            & (transitions["action"] == action)
        ]

        future_discomfort = sum(
            row.probability * speed_discomfort[row.next_state]
            for row in possible_transitions.itertuples(index=False)
        )

        # Measure ORIGINAL discomfort, rather than sick days
        new_discomfort[state] = (
            discomfort_lookup[(state, action)]
            + future_discomfort
        )

    largest_change = max(
        abs(new_discomfort[state] - speed_discomfort[state])
        for state in states
    )

    speed_discomfort = new_discomfort

    if largest_change < tolerance:
        print(f"Converged after {iteration + 1} iterations.")
        break
else:
    raise RuntimeError("Policy evaluation did not converge.")

# Compare with the minimum-discomfort policy evaluated in Part 3
policy_comparison = pd.DataFrame({
    "state": states,
    "speed_discomfort": [
        speed_discomfort[state] for state in states
    ],
    "minimize_discomfort": [
        policy_discomfort[state] for state in states
    ]
})

display(policy_comparison)

# The minimum-discomfort policy should have no higher discomfort
assert all(
    policy_discomfort[state] <= speed_discomfort[state] + 1e-9
    for state in states
)

Converged after 82 iterations.


,state,speed_discomfort,minimize_discomfort
0,exposed-1,0.833333,0.75
1,exposed-2,1.666667,1.50
2,exposed-3,3.333333,3.00
3,symptoms-1,6.666667,6.00
4,symptoms-2,5.000000,4.50
5,symptoms-3,1.666667,1.50
6,recovered,0.000000,0.00


Save the results to a file "policy-comparison.tsv" with columns state, speed_discomfort, and minimize_discomfort.

In [18]:
# YOUR CHANGES HERE

policy_comparison.to_csv(
    "policy-comparison.tsv",
    sep="\t",
    index=False
)

Submit "policy-comparison.tsv" in Gradescope.

## Part 8: Code

Please submit a Jupyter notebook that can reproduce all your calculations and recreate the previously submitted files.

## Part 9: Acknowledgements

If you discussed this assignment with anyone, please acknowledge them here.
If you did this assignment completely on your own, simply write none below.

If you used any libraries not mentioned in this module's content, please list them with a brief explanation what you used them for. If you did not use any other libraries, simply write none below.

If you used any generative AI tools, please add links to your transcripts below, and any other information that you feel is necessary to comply with the generative AI policy. If you did not use any generative AI tools, simply write none below.